In [2]:


import re
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from spellchecker import SpellChecker

In [3]:
df = pd.read_json("data/corpus.json")

# PDF text is full of newlines, so collapse all whitespace into single spaces
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True)
df["text_lower"] = df["text"].str.lower()   # used later for phrase matching

print(len(df), "documents loaded")

550 documents loaded


In [4]:
def tokenize(text):
    # lowercase handles case insensitivity, then keep only letters and digits
    return re.findall(r"[a-z0-9]+", text.lower())

# index the title together with the full text
tokenized_docs = [tokenize(t + " " + x) for t, x in zip(df["title"], df["text"])]
bm25 = BM25Okapi(tokenized_docs)

# every word seen in the corpus, used by the spell checker later
vocab = set(word for doc in tokenized_docs for word in doc)
print(len(vocab), "unique words")

102350 unique words


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# here we can use the full text, since there is no 256-token limit
dense_texts = (df["title"] + ". " + df["text"]).tolist()

# step 1: turn every document into a TF-IDF vector
tfidf = TfidfVectorizer(stop_words="english", max_features=50000, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(dense_texts)

# step 2: compress to 200 dense dimensions (LSA)
svd = TruncatedSVD(n_components=200, random_state=42)
doc_embeddings = normalize(svd.fit_transform(tfidf_matrix))   # unit length, so dot product = cosine

np.save("data/embeddings.npy", doc_embeddings)
print(doc_embeddings.shape)

def encode_query(text):
    # the query goes through the same two steps as the documents
    vec = svd.transform(tfidf.transform([text]))
    return normalize(vec)[0]

(550, 200)


In [6]:
ABBREVIATIONS = {
    "nlp": "natural language processing",
    "mt": "machine translation",
    "qa": "question answering",
    "llm": "large language model",
    "llms": "large language models",
    "nmt": "neural machine translation",
    "rl": "reinforcement learning",
    "rag": "retrieval augmented generation",
    "asr": "automatic speech recognition",
    "ner": "named entity recognition",
}

spell = SpellChecker()

def clean_query(query):
    # text inside "quotes" is a phrase that must appear exactly in the paper
    phrases = [p.lower() for p in re.findall(r'"([^"]+)"', query)]
    words = tokenize(query.replace('"', ''))

    final_words = []
    for w in words:
        if w in ABBREVIATIONS:
            final_words.append(w)                        # keep the abbreviation
            final_words.extend(tokenize(ABBREVIATIONS[w]))  # and add the full form
        elif w not in vocab:
            fixed = spell.correction(w)                  # only fix words the corpus has never seen
            final_words.append(fixed if fixed else w)
        else:
            final_words.append(w)
    return " ".join(final_words), phrases

In [7]:
def min_max(scores):
    # scale scores to 0-1 so BM25 and cosine scores are comparable
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

def reciprocal_rank_fusion(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for position, doc_id in enumerate(ranking):
            fused[doc_id] = fused.get(doc_id, 0) + 1 / (k + position + 1)
    return fused

In [8]:
def make_snippet(text, query_text, length=200):
    lower = text.lower()
    for w in query_text.split():
        if len(w) < 4:          # skip tiny words like "of" or "the"
            continue
        pos = lower.find(w)
        if pos != -1:
            start = max(0, pos - 50)
            return text[start:start + length]
    return text[:length]

def search(query, mode="rrf", top_k=10, alpha=0.5):
    query_text, phrases = clean_query(query)

    # score every document with both models
    bm25_scores = bm25.get_scores(tokenize(query_text))
    query_vec = encode_query(query_text)
    dense_scores = doc_embeddings @ query_vec

    if mode == "bm25":
        final = bm25_scores
    elif mode == "dense":
        final = dense_scores
    elif mode == "weighted":
        final = alpha * min_max(dense_scores) + (1 - alpha) * min_max(bm25_scores)
    elif mode == "rrf":
        fused = reciprocal_rank_fusion([np.argsort(-bm25_scores),
                                        np.argsort(-dense_scores)])
        final = np.array([fused[i] for i in range(len(df))])

    # phrase matching: drop any document that doesn't contain the quoted phrase
    for p in phrases:
        has_phrase = df["text_lower"].str.contains(p, regex=False).values
        final = np.where(has_phrase, final, -np.inf)

    results = []
    for i in np.argsort(-final)[:top_k]:
        if final[i] == -np.inf:
            continue
        results.append({
            "title": df["title"][i],
            "link": df["link"][i],
            "snippet": make_snippet(df["text"][i], query_text),
            "score": round(float(final[i]), 4),
        })
    return results

In [9]:
for mode in ["bm25", "dense", "weighted", "rrf"]:
    print("=====", mode)
    for r in search("NLP trasnlation of low resource languages", mode=mode, top_k=3):
        print(r["score"], r["title"])

print(search('"question answering" retrieval', top_k=2))

===== bm25
23.9759 Textual Augmentation Techniques Applied to Low Resource Machine Translation: Case of Swahili
23.9356 MahaSQuAD: Bridging Linguistic Divides in Marathi Question-Answering
23.8472 Robust Sentiment Analysis for Low Resource languages Using Data Augmentation Approaches: A Case Study in Marathi
===== dense
0.5521 Evaluating the Translation Performance of Large Language Models Based on Euas-20
0.5405 A Data Selection Approach for Enhancing Low Resource Machine Translation Using Cross-Lingual Sentence Representations
0.4702 MahaSQuAD: Bridging Linguistic Divides in Marathi Question-Answering
===== weighted
0.9257 MahaSQuAD: Bridging Linguistic Divides in Marathi Question-Answering
0.921 Evaluating the Translation Performance of Large Language Models Based on Euas-20
0.9118 A Data Selection Approach for Enhancing Low Resource Machine Translation Using Cross-Lingual Sentence Representations
===== rrf
0.032 MahaSQuAD: Bridging Linguistic Divides in Marathi Question-Answering
0

ui time


In [10]:
%pip install gradio

  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
    --------------------------------------- 0.5/31.4 MB 282.2 kB/s eta 0:01:50
    --------------------------------------- 0.5/31.4 MB 282.2 kB/s eta 0:01:50
    --------------------------------------- 0.5/31.4 MB 282.2 kB/s eta 0:01:50
    --------------------------------------- 0.5/31.4 MB 282.2 kB/s eta 0:01:50
    --------------------------------------- 0.

  You can safely remove it manually.


In [11]:
import gradio as gr

def gradio_search(query, mode, alpha):
    if query.strip() == "":
        return "Please type a query."

    results = search(query, mode=mode, top_k=10, alpha=alpha)
    cleaned, phrases = clean_query(query)
    text = f"Searching for: {cleaned} | phrases: {phrases}\n\n"

    if len(results) == 0:
        return text + "No results found."

    for rank, r in enumerate(results, start=1):
        text += f"**{rank}. {r['title']}** (score: {r['score']})\n\n"
        text += f"{r['link']}\n\n"
        text += f"{r['snippet']}...\n\n---\n\n"
    return text

demo = gr.Interface(
    fn=gradio_search,
    inputs=[
        gr.Textbox(label="Query"),
        gr.Dropdown(choices=[("Hybrid: RRF", "rrf"),
                             ("Hybrid: Weighted", "weighted"),
                             ("BM25 only", "bm25"),
                             ("Dense only", "dense")],
                    value="rrf", label="Model"),
        gr.Slider(0, 1, value=0.5, step=0.1,
                  label="Dense weight (weighted mode only)"),
    ],
    outputs=gr.Markdown(),
    title="arXiv cs.CL Hybrid Search",
)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
